# HiconaCooler

In [1]:
# Import packages
import os
import shutil
import tempfile

import polars as pl
import hicona as hi

# Data paths 
COOLER_PATH = "../data/IMR90_100kb_chr17_balanced.cool"
HMM_STATES = "../data/hmm_states_chr17.bed"
GENES = "../data/genes_chr17.bed"

## Reading data

As seen in the **Base Implementation** section, the usual entry point for HiCONA are `.cool` files. `HiconaCooler` is the class which allows you to interface with these files. To create a `HiconaCooler` object, simply pass the path to a valid cooler. If working with `.mcool` files, remember to add the full path to your resolution of interest (e.i. `somefile.mcool::resolutions/10000`).

In [2]:
handle = hi.HiconaCooler(COOLER_PATH)

As the name suggests, `HiconaCooler` inherits from the `Cooler` class from the `cooler` package, therefore retaining all its methods and attributes. As such, `HiconaCooler` objects can always be used in place of `Cooler` objects, including when reading data. For instance:

Since `HiconaCooler` inherits from `cooler.Cooler`, to fetch data you can use the `.bins()`, `.pixels()` and `.chroms()` methods. These methods return table selectors which can be sliced, subsetted and manipulated. Additionally `.matrix()` can be used to obtain the contact matrix in full matrix form. Here are some examples; for a full explanation refer to `cooler`'s documentation.


In [3]:
handle.pixels().fetch("chr17").head()

,bin1_id,bin2_id,count
0,26475,26475,17
1,26475,26481,1
2,26475,26518,1
3,26475,26526,1
4,26475,26534,1


In [4]:
handle.bins()[:5]

,chrom,start,end,weight
0,chr1,0,100000,NaN
1,chr1,100000,200000,NaN
2,chr1,200000,300000,NaN
3,chr1,300000,400000,NaN
4,chr1,400000,500000,NaN


In [5]:
handle.chroms()[:5]

,name,length
0,chr1,248956422
1,chr2,242193529
2,chr3,198295559
3,chr4,190214555
4,chr5,181538259


In [6]:
handle.matrix(balance=False).fetch("chr17:0-10000000")

array([[   17,     0,     0, ...,     0,     0,     0],
       [    0,  7035,  4003, ...,    14,    13,    14],
       [    0,  4003, 11872, ...,    23,    29,    13],
       ...,
       [    0,    14,    23, ..., 11189,  4187,  2267],
       [    0,    13,    29, ...,  4187, 12808,  3767],
       [    0,    14,    13, ...,  2267,  3767, 14430]], dtype=int32)

These selectors work well when you are interested in pixels as they are (for plotting purposes for instance) and when your pixels of interest are contiguous is the table. When you need a complex selection or applying operations which would require chunked reading multiple times, you are better off creating a `PixelTable` and working on that.

In [7]:
# Some arbitrary lambda function to apply on pixel chunks
count_filter = lambda x: (y.filter(pl.col("count") > 3) for y in x)         # Remove pixels with count <= 3
count_scaler = lambda x: (y.with_columns(pl.col("count") * 10) for y in x)  # Multiply counts by 10

pix_table = handle.get_pixel_table("chr17").apply((count_filter, count_scaler))
pix_table.get_dataframe().head(10)

bin1_id,bin2_id,count
i64,i64,i32
26475,26475,170
26476,26476,70350
26476,26477,40030
26476,26478,33920
26476,26479,11320
26476,26480,8680
26476,26481,11400
26476,26482,14960
26476,26483,3700


Creating a `PixelTable` from a `.cool` file has some overhead which can be fairly big for huge files (10s of seconds), though it still leads to a significant speedup when you need to apply several operations to an arbitrary subset of the pixels. More on `PixelTable` objects in the dedicated notebook.

## Bin annotation

`Cooler` does not allow to easily add bin and pixel annotations to a file, unless they were provided at file creation. The `HiconaCooler` class provides a function, `bin_annotation_add`, to add one or more bin annotations to a previously generated cooler file. Different combinations of the parameters (`metric`, `consolidate`, `save_all_mods`, `ignore_null_mode`) can be used depending on the type of annotation to add. Here are some of the most likely scenarios you might encounter:

- **gene annotation (permissive)**: annotate each bin with the gene overlapping the most with it. Only annotate with `None` if there is absolutely no overlapping gene. Better suited when working with higher resolutions (10 kb or more).
- **gene annotation (restrictive)**: annotate each bin with the gene overlapping the most with it. Annotate with `None` if the fraction of non gene-overlapping bases exceeds the number of bases of the gene with the biggest overlap. Better suited for smaller resolutions (less than 10 kb).
- **chromHMM-like annotation (most)**: given an annotation with few repetitive modalities covering the genome (e.i. promoter, enhancer, euchromatin, heterochromatin...), annotate each bin with the modality which is most enriched with respect to the chromosome containing the bin. Enrichment for an annotation is defined as log fold change between fraction of the bin with the annotation and fraction of the chromosome with the annotation. `None` is considered a valid modality, therefore it is the chosen annotation if it is the most enriched.
- **chromHMM-like annotation (all)**: similar to the previous point, but instead create an annotation column for each modality (null modality included) and save the enrichment of the modality in each bin. This can be quite **heavy on file size**.
  
Here are the combinations of parameters for these modalities:

|                                 | metric         | consolidate | save_all_mods | ignore_null_mode |
|---------------------------------|----------------|-------------|---------------|------------------|
| gene annotation (permissive)    | "bp_overlap"   | False       | False         | True             |
| gene annotation (restrictive)   | "bp_overlap"   | True        | False         | False            |
| chromHMM-like annotation (most) | "chrom_enrich" | True        | False         | False            |
| chromHMM-like annotation (all)  | "chrom_enrich" | True        | True          | False            |


<div class="alert alert-warning">

Annotations can make the size of the cooler increase by a lot very quickly. In general, it is better to add the annotation to the cooler only when you are likely to use it multiple times. For one-time annotations, HiCONA allows to annotate `BinTable`/`PixelTable` objects directly.

</div>

<div class=note>
Using gene annotations with resolutions above 10kb is, in general, not a great idea. Here it is done just for demonstration purposes.
</div>

**Example**: gene annotation (permissive)

In [8]:
# Bed file containing gene information
genes = pl.read_csv(GENES, separator="\t")
genes.head()

chrom,start,end,gene
str,i64,i64,str
"""chr17""",64098,76866,"""ENSG00000280279.1"""
"""chr17""",88640,118578,"""ENSG00000280136.2"""
"""chr17""",90795,90899,"""ENSG00000277613.1"""
"""chr17""",137568,139067,"""ENSG00000268320.4"""
"""chr17""",142788,181650,"""ENSG00000272636.4"""


In [9]:
# Work on a temporary copy of the file to avoid permanent changes
with tempfile.TemporaryDirectory() as tmp_dir_name:
    tmp_file = os.path.join(tmp_dir_name, os.path.basename(COOLER_PATH))
    shutil.copy2(COOLER_PATH, tmp_file)

    copy_handle = hi.HiconaCooler(tmp_file)
    copy_handle.bin_annot_add(
        genes, 
        metric="bp_overlap", 
        consolidate=False,
        save_all_mods=False,
        ignore_null_mode=True,
    )
    
    annot_bins = copy_handle.bins().fetch("chr17")

annot_bins.head()

,chrom,start,end,gene,weight
26475,chr17,0,100000,ENSG00000280279.1,NaN
26476,chr17,100000,200000,ENSG00000272636.4,0.007338
26477,chr17,200000,300000,ENSG00000181031.16,0.006855
26478,chr17,300000,400000,ENSG00000181031.16,0.005692
26479,chr17,400000,500000,ENSG00000187624.9,0.007215


**Example**: chromHMM-like annotation (all)

In [10]:
# ChromHMM like-annotation
hmm_states = pl.read_csv(HMM_STATES, separator="\t", new_columns=("chrom", "start", "end", "state"))
hmm_states.head()

chrom,start,end,state
str,i64,i64,str
"""chr17""",150207,152007,"""Repr_Pc"""
"""chr17""",152007,157009,"""Repr_Pc"""
"""chr17""",157009,166609,"""Repr_Pc"""
"""chr17""",166609,172409,"""Repr_Pc"""
"""chr17""",172409,181809,"""Repr_Pc"""


In [11]:
# Work on a temporary copy of the file to avoid permanent changes
with tempfile.TemporaryDirectory() as tmp_dir_name:
    tmp_file = os.path.join(tmp_dir_name, os.path.basename(COOLER_PATH))
    shutil.copy2(COOLER_PATH, tmp_file)

    copy_handle = hi.HiconaCooler(tmp_file)
    copy_handle.bin_annot_add(
        hmm_states, 
        metric="chrom_enrich", 
        consolidate=True, 
        save_all_mods=True, 
        ignore_null_mode=False,
    )
    annot_bins = copy_handle.bins().fetch("chr17")

annot_bins

,chrom,start,end,Enh_chrom_enrich,Het_chrom_enrich,Prom_chrom_enrich,Repr_Pc_chrom_enrich,Tx_chrom_enrich,Void_chrom_enrich,null_chrom_enrich,weight
26475,chr17,0,100000,NaN,NaN,NaN,NaN,NaN,NaN,4.034009,NaN
26476,chr17,100000,200000,0.035446,0.286166,NaN,1.487346,NaN,NaN,3.124769,0.007338
26477,chr17,200000,300000,0.070042,1.360146,NaN,2.029982,NaN,0.425676,NaN,0.006855
26478,chr17,300000,400000,0.903479,1.428936,0.788412,1.928669,0.058686,0.263276,NaN,0.005692
26479,chr17,400000,500000,0.508783,0.616187,0.719294,0.616640,0.521980,0.893754,3.034053,0.007215
...,...,...,...,...,...,...,...,...,...,...,...
27303,chr17,82800000,82900000,1.603893,0.616187,0.404018,NaN,1.905742,NaN,NaN,0.005922
27304,chr17,82900000,83000000,0.783524,1.945662,0.313119,NaN,1.516685,1.021199,NaN,0.006316
27305,chr17,83000000,83100000,1.661746,0.809179,2.017233,0.806154,1.437483,NaN,0.181779,0.006349
27306,chr17,83100000,83200000,1.823376,0.674011,0.666660,1.873460,0.058686,NaN,0.378277,0.006347


<div class=note>
`HiconaCooler` also provide a `bin_annot_del` method to remove bin annotations. This is useful to cleanup the column namespace and avoid confusion. It must be noted though that, due to how the `.hdf5` file format works, this does not reduce file size. To reduce file size after removing an annotation you must use some external tool such as `h5repack`.
</div>

In [12]:
from IPython.core.display import HTML

def css_styling():
    styles = open("../styles/custom.css", "r").read()
    return HTML(styles)
css_styling()